# Imports & code

In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from resin_spectral_analysis.process_absorber_spectrum import process_spectrum_data, read_spectrum_config_json
from resin_spectral_analysis.utilities import read_OE65PRO_spectrum_data_file, find_index_of_nearest

In [2]:
# def read_spectrum_config_json(directory, spectrum_config_file="spectrum_config.json"):
#     assert isinstance(directory, Path)
#     with (directory / spectrum_config_file).open() as f:
#          temp = json.load(f)
#     return temp


# def find_indices_of_range(arr, low, high):
#     assert len(arr.shape) == 1
    
#     index_low = find_index_of_nearest(arr, low)
#     index_high = find_index_of_nearest(arr, high)
    
#     return index_low, index_high


# def select_wavelength_range(data, wavelength_low, wavelength_high):
#     assert len(data.shape) == 2
#     assert data.shape[1] == 2
    
#     index_low, index_high = find_indices_of_range(data[:,0], wavelength_low, wavelength_high)
    
#     return data[index_low:index_high+1]


# def calc_absorbance(source_spectrum, absorber_spectrum, wavelength_low, wavelength_high, force_zero=True, num_samples_for_average=10):
    
#     # Make sure wavelengths are the same for both data sets
#     assert np.allclose(source_spectrum[:,0], absorber_spectrum[:,0])
    
#     # Select only the wavelength range we need
#     temp_source = select_wavelength_range(source_spectrum, wavelength_low, wavelength_high)
#     temp_absorber = select_wavelength_range(absorber_spectrum, wavelength_low, wavelength_high)
    
#     # Calculate absorbance
#     absorbance = np.column_stack((temp_source[:,0], -np.log10(temp_absorber[:,1] / temp_source[:,1])))
    
#     # Make sure the average absorbance is 0 in the high wavelength range
#     # where there is presumably no absorption
#     if force_zero:
#         avg = np.mean(absorbance[-num_samples_for_average:, 1])
#         if avg < 0.0:
#             absorbance[:, 1] += -avg
        
#     return absorbance


# def calc_molar_concentration(concentration_w_w_percent, density_of_solute_g_per_L, molar_mass_g_per_mole):
#     """Calculates molar concentration in moles per Liter"""
#     return (concentration_w_w_percent / 100.) * (density_of_solute_g_per_L / molar_mass_g_per_mole)


# def calc_molar_absorptivity(absorbance, molar_concentration, length_um):
#     length_cm = 1.e-4 * length_um
#     return absorbance / (molar_concentration * length_cm)


# def process_spectrum_data(directory, spectrum_config, wavelength_range=[300, 540]):
#     """Process spectrometer data to create absorbance and molar absorptivity csv files.
    
#     Parameters
#     ----------
#     directory - Path object
#         Directory in which `raw_data` folder is present, and in which the 2 output files
#         will be written.
#     spectrum_config - dict
#         Specify files and parameters needed to create absorbance and molar absorptivity.
#     wavelength_range - list
#         Wavelength range over which to do calculations. Low end is set by signal to noise
#         ratio of data, and high end is larger than 532 nm laser wavelength, but smaller than
#         saturated measurement data starting at 543 nm.
        
#     Returns
#     -------
#     Nothing
    
#     Outputs
#     -------
#     absorbance.csv file
#     molar_absorptivity.csv file
#     """
    
#     assert isinstance(directory, Path)
#     assert isinstance(spectrum_config, dict)
    
#     # Check that parameters required for absorbance calculation are in spectrum_config
#     required_params_for_absorbance_calc = [
#         'Solvent absorption measurement file',
#         'Solvent + absorber absorption measurement file',
#     ]
#     for p in required_params_for_absorbance_calc:
#         if p not in spectrum_config:
#             msg = f"Required parameter '{p}' for absorbance calculation is not in 'spectrum_config.json'"
#             raise KeyError(msg)
    
#     raw_data_directory = directory / "raw_data"
    
#     # Read solvent data
#     solvent_file = raw_data_directory / spectrum_config["Solvent absorption measurement file"]
#     solvent_spectrometer_header, solvent_spectrometer_data = read_OE65PRO_spectrum_data_file(solvent_file)
    
#     # Read solvent + absorber data
#     solvent_and_absorber_file = raw_data_directory / spectrum_config["Solvent + absorber absorption measurement file"]
#     solvent_and_absorber_spectrometer_header, solvent_and_absorber_spectrometer_data = read_OE65PRO_spectrum_data_file(solvent_and_absorber_file)
        
#     # Create absorbance data & write to file
#     absorbance = calc_absorbance(solvent_spectrometer_data, solvent_and_absorber_spectrometer_data, *wavelength_range)
#     output_data_file = directory / "absorbance.csv"
#     np.savetxt(output_data_file, absorbance, delimiter=",", header=json.dumps(spectrum_config))
    
#     # Check that parameters required for molar absorptivity calculation are in spectrum_config
#     required_params_for_molar_absorptivity_calc = [
#         'Concentration w/w percent',
#         'Film thickness microns',
#         'Molar mass g/mole',
#         'Solvent density g/L',
#     ]
#     for p in required_params_for_molar_absorptivity_calc:
#         if p not in spectrum_config:
#             msg = f"Required parameter '{p}' for molar absorptivity calculation is not in 'spectrum_config.json'."
#             msg += "\n'molar_absorptivity.csv` file not created."
#             sys.exit()
            
#     # Create molar absorbance data & write to file
#     molar_concentration = calc_molar_concentration(
#         spectrum_config['Concentration w/w percent'], 
#         spectrum_config['Solvent density g/L'], 
#         spectrum_config['Molar mass g/mole']
#     )
#     molar_absorptivity = absorbance.copy()
#     molar_absorptivity[:, 1] = calc_molar_absorptivity(
#         absorbance[:, 1], molar_concentration, spectrum_config['Film thickness microns']
#     )
#     output_data_file = directory / "molar_absorptivity.csv"
#     np.savetxt(output_data_file, molar_absorptivity, delimiter=",", header=json.dumps(spectrum_config))    
    

# Set up directory and read `spectrum_config.json`

In [3]:
directory = Path(".")
print(directory.resolve())

spectrum_config = read_spectrum_config_json(directory.resolve())
spectrum_config

/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/resin_spectral_analysis/data/absorbers/avobenzone_in_PEGDA_2020-01-20


{'Measurement': 'Avobenzone absorption in PEGDA',
 'Measurement Date': '2020-01-20',
 'Measurement Spectrometer': 'Ocean Optics FIRE',
 'Absorber': 'Avobenzone',
 'Concentration w/w percent': 0.24,
 'Film thickness microns': 60,
 'Molar mass g/mole': 310.39,
 'Solvent': 'PEGDA',
 'Solvent density g/L': 1110,
 'Solvent absorption measurement file': 'QEPB00791_17-10-07-458.txt',
 'Solvent + absorber absorption measurement file': 'QEPB00791_17-18-57-140.txt'}

# Process spectrum data to create absorbance & molar absorptivity

In [6]:
process_spectrum_data(directory.resolve(), spectrum_config)

# Plots

## Absorbance

In [7]:
absorbance = np.loadtxt(directory / "absorbance.csv", delimiter=',')

fig, ax = plt.subplots()
ax.plot(absorbance[:,0], absorbance[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
title_text = spectrum_config['Absorber'] + ', ' + spectrum_config['Measurement Date']
ax.set_title(title_text)
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Molar absorptivity

In [8]:
molar_absorptivity = np.loadtxt(directory / "molar_absorptivity.csv", delimiter=',')

fig, ax = plt.subplots()
ax.plot(molar_absorptivity[:,0], molar_absorptivity[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Molar Absorptivity (L $\cdot$ mol$^{-1} \cdot$cm$^{-1}$)')
title_text = spectrum_config['Absorber'] + ', ' + spectrum_config['Measurement Date']
ax.set_title(title_text)
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Next

- Put code in a utilities module and test
- Use code to analyze 2017 avobenzone absorption measurements and compare to 2020 measurements
- Do the same for Irgacure 819 photoinitiator as for avobenzone
- Document process of adding a photoinitiator or absorber